In [6]:
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier


In [7]:
# 모델별 저장 경로
MODEL_DIRS = {
    'swin': '/nahcooy/OSR/classify/ensemble/logits/swin',
    'convnext': '/nahcooy/OSR/classify/ensemble/logits/convnext',
    'yolo': '/nahcooy/OSR/classify/ensemble/logits/Yolo'
}

# split: 'train' or 'val'
def load_combined_logits(split='train'):
    logits_list = []
    labels = None

    for name, dir_path in MODEL_DIRS.items():
        logits = np.load(os.path.join(dir_path, f'{split}_logits.npy'))
        logits_list.append(logits)

        if labels is None:
            labels = np.load(os.path.join(dir_path, f'{split}_labels.npy'))

    combined_logits = np.concatenate(logits_list, axis=1)
    return combined_logits, labels


In [8]:
X_train, y_train = load_combined_logits('train')
X_val, y_val = load_combined_logits('val')

print("🔹 X_train shape:", X_train.shape)
print("🔹 y_train shape:", y_train.shape)


🔹 X_train shape: (16702, 21)
🔹 y_train shape: (16702,)


In [9]:
# 🔹 사용할 sklearn 기반 모델 정의
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=10),
    "SVM": SVC(probability=True),
    "MLP": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500)
}

results = {}

# 🔸 학습 및 평가 루프
for name, model in models.items():
    print(f"\n🔸 Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')

    results[name] = {'accuracy': acc, 'f1_macro': f1}
    print(f"✅ {name} | Acc: {acc:.4f} | F1 (macro): {f1:.4f}")


🔸 Training LogisticRegression...
✅ LogisticRegression | Acc: 0.9311 | F1 (macro): 0.9335

🔸 Training RandomForest...
✅ RandomForest | Acc: 0.9321 | F1 (macro): 0.9347

🔸 Training SVM...
✅ SVM | Acc: 0.9431 | F1 (macro): 0.9446

🔸 Training MLP...
✅ MLP | Acc: 0.9356 | F1 (macro): 0.9375


In [5]:
import pandas as pd

result_df = pd.DataFrame(results).T
result_df.sort_values('f1_macro', ascending=False)


,accuracy,f1_macro
SVM,0.943085,0.944577
MLP,0.937094,0.939485
LogisticRegression,0.931103,0.933508
RandomForest,0.930105,0.932400


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# 🔹 로짓 로딩 함수
MODEL_DIRS = {
    'swin': '/nahcooy/OSR/classify/ensemble/logits/swin',
    'convnext': '/nahcooy/OSR/classify/ensemble/logits/convnext',
    'yolo': '/nahcooy/OSR/classify/ensemble/logits/Yolo'
}

def load_combined_logits(split='train'):
    logits_list = []
    labels = None
    for path in MODEL_DIRS.values():
        logits = np.load(f"{path}/{split}_logits.npy")
        logits_list.append(logits)
        if labels is None:
            labels = np.load(f"{path}/{split}_labels.npy")
    return np.concatenate(logits_list, axis=1), labels

# 🔹 데이터 로드
X_train, y_train = load_combined_logits('train')
X_val, y_val = load_combined_logits('val')

# 🔸 모델 정의
models = {
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='mlogloss',
        tree_method='gpu_hist',        # GPU 사용 가능 (필요시 제거 가능)
        predictor='gpu_predictor'
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        num_leaves=7,
        min_data_in_leaf=5,
        min_child_samples=10,
        device_type='cpu'  # GPU 비활성화
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.1
    )
}

# 🔸 학습 및 평가
results = {}

for name, model in models.items():
    print(f"\n🔸 Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')

    results[name] = {'accuracy': acc, 'f1_macro': f1}
    print(f"✅ {name} | Accuracy: {acc:.4f} | F1 (macro): {f1:.4f}")



🔸 Training XGBoost...


/home/nahcooy/miniconda3/envs/osr/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [14:38:56] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/nahcooy/miniconda3/envs/osr/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [14:38:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor", "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost | Accuracy: 0.9306 | F1 (macro): 0.9202

🔸 Training LightGBM...
[LightGBM] [Warning] min_data_in_leaf is set=1, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1
[LightGBM] [Warning] min_data_in_leaf is set=1, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 5353
[LightGBM] [Info] Number of data points in the train set: 16702, number of used features: 21
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3090 Ti, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 21 dense feature groups (0.38 MB) transferred to GPU in 0.003082 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.135819
[LightGBM] [Info] Start training from score -2.240602
[LightGBM] [Info] Start training from score -2.2

[LightGBM] [Fatal] Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .



LightGBMError: Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .
